# Insolvenzvorhersage polnischer Unternehmen mittels Machine Learning

Binaere Klassifikation auf Basis von Jahresabschlussdaten des polnischen Markts (2000 bis 2013)

In [ ]:
%pip install pandas numpy scipy scikit-learn plotly xgboost -q

In [ ]:
import pandas as pd
import numpy as np
from scipy.io import arff
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data'

feature_names = {
    'Attr1':  'X1_net_profit_total_assets',
    'Attr2':  'X2_total_liabilities_total_assets',
    'Attr3':  'X3_working_capital_total_assets',
    'Attr4':  'X4_current_assets_short_term_liabilities',
    'Attr5':  'X5_cash_flow_days',
    'Attr6':  'X6_retained_earnings_total_assets',
    'Attr7':  'X7_EBIT_total_assets',
    'Attr8':  'X8_book_value_equity_total_liabilities',
    'Attr9':  'X9_sales_total_assets',
    'Attr10': 'X10_equity_total_assets',
    'Attr11': 'X11_gross_profit_extraord_finexp_total_assets',
    'Attr12': 'X12_gross_profit_short_term_liabilities',
    'Attr13': 'X13_gross_profit_depr_sales',
    'Attr14': 'X14_gross_profit_interest_total_assets',
    'Attr15': 'X15_total_liabilities_days_gross_profit_depr',
    'Attr16': 'X16_gross_profit_depr_total_liabilities',
    'Attr17': 'X17_total_assets_total_liabilities',
    'Attr18': 'X18_gross_profit_total_assets',
    'Attr19': 'X19_gross_profit_sales',
    'Attr20': 'X20_inventory_days_sales',
    'Attr21': 'X21_sales_growth',
    'Attr22': 'X22_operating_profit_total_assets',
    'Attr23': 'X23_net_profit_sales',
    'Attr24': 'X24_gross_profit_3y_total_assets',
    'Attr25': 'X25_equity_share_capital_total_assets',
    'Attr26': 'X26_net_profit_depr_total_liabilities',
    'Attr27': 'X27_operating_profit_financial_expenses',
    'Attr28': 'X28_working_capital_fixed_assets',
    'Attr29': 'X29_log_total_assets',
    'Attr30': 'X30_total_liabilities_cash_sales',
    'Attr31': 'X31_gross_profit_interest_sales',
    'Attr32': 'X32_current_liabilities_days_cost_products',
    'Attr33': 'X33_operating_expenses_short_term_liabilities',
    'Attr34': 'X34_operating_expenses_total_liabilities',
    'Attr35': 'X35_profit_on_sales_total_assets',
    'Attr36': 'X36_total_sales_total_assets',
    'Attr37': 'X37_current_assets_inv_long_term_liabilities',
    'Attr38': 'X38_constant_capital_total_assets',
    'Attr39': 'X39_profit_on_sales_sales',
    'Attr40': 'X40_current_assets_inv_rec_short_term_liab',
    'Attr41': 'X41_total_liabilities_operating_days',
    'Attr42': 'X42_operating_profit_sales',
    'Attr43': 'X43_receivables_inventory_turnover_days',
    'Attr44': 'X44_receivables_days_sales',
    'Attr45': 'X45_net_profit_inventory',
    'Attr46': 'X46_current_assets_inv_short_term_liabilities',
    'Attr47': 'X47_inventory_days_cost_products',
    'Attr48': 'X48_EBITDA_total_assets',
    'Attr49': 'X49_EBITDA_sales',
    'Attr50': 'X50_current_assets_total_liabilities',
    'Attr51': 'X51_short_term_liabilities_total_assets',
    'Attr52': 'X52_short_term_liabilities_days_cost_products',
    'Attr53': 'X53_equity_fixed_assets',
    'Attr54': 'X54_constant_capital_fixed_assets',
    'Attr55': 'X55_working_capital',
    'Attr56': 'X56_sales_gross_margin',
    'Attr57': 'X57_liquidity_ratio',
    'Attr58': 'X58_total_costs_total_sales',
    'Attr59': 'X59_long_term_liabilities_equity',
    'Attr60': 'X60_sales_inventory',
    'Attr61': 'X61_sales_receivables',
    'Attr62': 'X62_short_term_liabilities_days_sales',
    'Attr63': 'X63_sales_short_term_liabilities',
    'Attr64': 'X64_sales_fixed_assets',
}

dfs = []
for year in range(1, 6):
    data, meta = arff.loadarff(f'{DATA_DIR}/{year}year.arff')
    df = pd.DataFrame(data)
    df['year'] = year
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
df_all['class'] = df_all['class'].apply(lambda x: int(x.decode()) if isinstance(x, bytes) else int(x))
df_all.rename(columns=feature_names, inplace=True)

feat_cols = [c for c in df_all.columns if c not in ['class', 'year']]

print(f'Gesamtform: {df_all.shape}')
print(f'Klassen-Verteilung:\n{df_all["class"].value_counts()}')

## 1. Einleitung und Problemdefinition

Insolvenz ist ein zentrales Risiko im wirtschaftlichen Umfeld und betrifft Glaeubiger, Investoren, Mitarbeiter und die Gesamtwirtschaft gleichermassen. Die fruehzeitige Erkennung von Insolvenzrisiken ist daher von erheblicher praktischer Bedeutung fuer Kreditinstitute, Wirtschaftspruefer und Investoren.

Traditionale statistische Modelle wie der Z-Score nach Altman (1968) oder das logistische Regressionsmodell nach Ohlson (1980) haben gezeigt, dass Finanzkennzahlen aus Jahresabschluessen zur Insolvenzvorhersage geeignet sind. Mit der zunehmenden Verfuegbarkeit rechenstarker Algorithmen ruecken Machine-Learning-Verfahren in den Vordergrund, die komplexe, nichtlineare Zusammenhaenge in den Daten erkennen koennen.

**Wissenschaftliche Fragestellung:**

Koennen Insolvenzrisiken polnischer Unternehmen auf Basis von Jahresabschlussdaten mithilfe von Machine-Learning-Klassifikationsmodellen zuverlaessig vorhergesagt werden, und wie veraendert sich die Vorhersagequalitaet in Abhaengigkeit vom Prognosehorizont?

**Teilfragen:**

1. Welches Klassifikationsmodell (Logistische Regression, Random Forest, XGBoost) erzielt die beste Vorhersageleistung?
2. Wie wirkt sich der Prognosehorizont (1 bis 5 Jahre vor Insolvenz) auf die Modellguete aus?
3. Welche Finanzkennzahlen sind fuer die Insolvenzvorhersage am bedeutsamsten?

## 2. Theoretischer Hintergrund

### 2.1 Klassische Modelle der Insolvenzvorhersage

Die wissenschaftliche Auseinandersetzung mit Insolvenzvorhersage reicht bis in die 1960er Jahre zurueck. Beaver (1966) zeigte in einer univariaten Analyse, dass einzelne Finanzkennzahlen wie der Cash-Flow-to-Debt-Ratio signifikante Vorhersagekraft besitzen. Altman (1968) entwickelte mit dem Z-Score ein multivariates diskriminanzanalytisches Modell, das fuenf Finanzkennzahlen kombiniert und bis heute als Referenzmodell gilt. Ohlson (1980) fuehrte die logistische Regression in die Insolvenzforschung ein und ermoeglichte damit die direkte Schaetzung von Insolvenzwahrscheinlichkeiten.

### 2.2 Machine Learning in der Insolvenzvorhersage

Barboza et al. (2017) verglichen klassische statistische Verfahren mit modernen Machine-Learning-Methoden (SVM, Bagging, Boosting, Random Forest) und zeigten, dass baumbasierte Ensemble-Verfahren die tradierten Modelle konsistent uebertreffen. Liang et al. (2016) bestaetigen die Ueberlegenheit von Gradient-Boosting-Verfahren insbesondere bei unbalancierten Datensaetzen mit vielen Finanzkennzahlen.

### 2.3 Datensatz

Der verwendete Datensatz stammt von Zieba et al. (2016) und umfasst Finanzdaten polnischer Unternehmen aus der EMIS-Datenbank (Emerging Markets Information Service) fuer den Zeitraum 2000 bis 2013. Er enthaelt 64 Finanzkennzahlen zu Profitabilitaet, Liquiditaet, Verschuldung, Kapitalstruktur und Aktivitaet. Die fuenf Teildatensaetze unterscheiden sich im Prognosehorizont: Datensatz 1 erlaubt eine Vorhersage 5 Jahre vor Insolvenz, Datensatz 5 eine Vorhersage 1 Jahr vor Insolvenz.

**Literatur:**

- Altman, E.I. (1968). Financial ratios, discriminant analysis and the prediction of corporate bankruptcy. Journal of Finance, 23(4), 589-609.
- Barboza, F., Kimura, H., & Altman, E. (2017). Machine learning models and bankruptcy prediction. Expert Systems with Applications, 83, 405-417.
- Beaver, W.H. (1966). Financial ratios as predictors of failure. Journal of Accounting Research, 4, 71-111.
- Liang, D., Lu, C.C., Tsai, C.F., & Shih, G.A. (2016). Financial ratios and corporate governance indicators in bankruptcy prediction. Expert Systems with Applications, 52, 227-240.
- Ohlson, J.A. (1980). Financial ratios and the probabilistic prediction of bankruptcy. Journal of Accounting Research, 18(1), 109-131.
- Zieba, M., Tomczak, S.K., & Tomczak, J.M. (2016). Ensemble boosted trees with synthetic features generation in application to bankruptcy prediction. Expert Systems with Applications, 58, 93-101.

## 3. Datenbasis

### 3.1 Beschreibung des Datensatzes

Der Datensatz besteht aus fuenf ARFF-Dateien (1year.arff bis 5year.arff), die jeweils einen anderen Prognosehorizont repraesentieren. Die Zielvariable ist binaer (0 = nicht bankrott, 1 = bankrott). Die folgende Tabelle zeigt die Verteilung:

| Datei | Prognosehorizont | Gesamt | Bankrott | Nicht bankrott |
|---|---|---|---|---|
| 1year.arff | 5 Jahre | 7.027 | 271 | 6.756 |
| 2year.arff | 4 Jahre | 10.173 | 400 | 9.773 |
| 3year.arff | 3 Jahre | 10.503 | 495 | 10.008 |
| 4year.arff | 2 Jahre | 9.792 | 515 | 9.277 |
| 5year.arff | 1 Jahr | 5.910 | 410 | 5.500 |

Die Merkmale decken die folgenden Kategorien ab: Profitabilitaet (z.B. X1, X7, X18), Verschuldung (X2, X17), Liquiditaet (X3, X4), Kapitalstruktur (X10, X38) sowie Aktivitaets- und Umschlagskennzahlen (X9, X36).

### 3.2 Explorative Datenanalyse

Die explorative Datenanalyse (EDA) wird ausschliesslich zur Gewinnung eines Ueberblicks ueber die Datenstruktur eingesetzt. Saemtliche Vorverarbeitungsschritte (Imputation, Skalierung) werden spaeter ausschliesslich auf dem Trainingsset berechnet, um Data Leakage zu vermeiden.

In [ ]:
print('Grundlegende Statistiken:')
print(f'Anzahl Beobachtungen gesamt: {len(df_all)}')
print(f'Anzahl Merkmale: {len(feat_cols)}')
print(f'Fehlende Werte gesamt: {df_all[feat_cols].isnull().sum().sum()}')
print()
print('Klassen-Verteilung pro Jahr:')
print(df_all.groupby('year')['class'].value_counts().unstack(fill_value=0))

In [ ]:
class_counts = df_all['class'].value_counts().reset_index()
class_counts.columns = ['class', 'count']
class_counts['Klasse'] = class_counts['class'].map({0: 'Nicht bankrott', 1: 'Bankrott'})

fig = px.pie(
    class_counts,
    names='Klasse',
    values='count',
    color='Klasse',
    color_discrete_map={'Nicht bankrott': '#2196F3', 'Bankrott': '#F44336'},
    hole=0.4,
    title='Abbildung 1: Gesamtverteilung der Zielklassen'
)
fig.show()

In [ ]:
year_stats = df_all.groupby('year')['class'].agg(['sum', 'count']).reset_index()
year_stats['Bankrott-Rate (%)'] = (year_stats['sum'] / year_stats['count'] * 100).round(2)
year_stats['Prognosehorizont'] = year_stats['year'].map(
    {1: '5 Jahre', 2: '4 Jahre', 3: '3 Jahre', 4: '2 Jahre', 5: '1 Jahr'})

fig = px.bar(
    year_stats,
    x='Prognosehorizont',
    y='Bankrott-Rate (%)',
    text='Bankrott-Rate (%)',
    color='Bankrott-Rate (%)',
    color_continuous_scale='Reds',
    title='Abbildung 2: Bankrott-Rate je Prognosehorizont'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_coloraxes(showscale=False)
fig.show()

In [ ]:
missing_pct = (df_all[feat_cols].isnull().sum() / len(df_all) * 100).sort_values(ascending=True)
missing_pct = missing_pct[missing_pct > 0].reset_index()
missing_pct.columns = ['Feature', 'Fehlend (%)']
missing_pct['Feature_kurz'] = missing_pct['Feature'].str.split('_').str[0]

fig = px.bar(
    missing_pct,
    x='Fehlend (%)',
    y='Feature_kurz',
    orientation='h',
    color='Fehlend (%)',
    color_continuous_scale='Reds',
    text='Fehlend (%)',
    title='Abbildung 3: Anteil fehlender Werte pro Merkmal'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=700, yaxis_title='Merkmal', coloraxis_showscale=False)
fig.show()

In [ ]:
medians = df_all.groupby('class')[feat_cols].median().T
medians.columns = ['Nicht bankrott', 'Bankrott']
medians['diff_norm'] = (medians['Bankrott'] - medians['Nicht bankrott']) / df_all[feat_cols].std()
top20 = medians.reindex(medians['diff_norm'].abs().sort_values(ascending=True).index).tail(20).reset_index()
top20.columns = ['Feature', 'Nicht bankrott', 'Bankrott', 'Norm. Differenz']
top20['Feature_kurz'] = top20['Feature'].str.split('_').str[0]

fig = px.bar(
    top20,
    x='Norm. Differenz',
    y='Feature_kurz',
    color='Norm. Differenz',
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0,
    orientation='h',
    title='Abbildung 4: Top-20-Merkmale nach normierter Median-Differenz (Bankrott vs. Nicht bankrott)'
)
fig.update_layout(height=550, yaxis_title='Merkmal', coloraxis_showscale=False)
fig.show()

In [ ]:
corr_features = [
    'X1_net_profit_total_assets', 'X2_total_liabilities_total_assets',
    'X3_working_capital_total_assets', 'X7_EBIT_total_assets',
    'X10_equity_total_assets', 'X18_gross_profit_total_assets',
    'X22_operating_profit_total_assets', 'X25_equity_share_capital_total_assets',
    'X29_log_total_assets', 'X35_profit_on_sales_total_assets',
    'X48_EBITDA_total_assets', 'X51_short_term_liabilities_total_assets',
    'class'
]

corr = df_all[corr_features].corr()
corr.index   = [c.split('_')[0] for c in corr.index]
corr.columns = [c.split('_')[0] for c in corr.columns]

fig = px.imshow(
    corr,
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0,
    zmin=-1, zmax=1,
    text_auto='.2f',
    title='Abbildung 5: Pearson-Korrelationsmatrix ausgewaehlter Merkmale'
)
fig.update_layout(height=600, width=700)
fig.show()

In [ ]:
q1 = df_all[feat_cols].quantile(0.25)
q3 = df_all[feat_cols].quantile(0.75)
iqr = q3 - q1
outlier_mask = (df_all[feat_cols] < (q1 - 3 * iqr)) | (df_all[feat_cols] > (q3 + 3 * iqr))
outlier_pct = (outlier_mask.sum() / len(df_all) * 100).sort_values(ascending=False).head(20).reset_index()
outlier_pct.columns = ['Feature', 'Ausreisser (%)']
outlier_pct['Feature_kurz'] = outlier_pct['Feature'].str.split('_').str[0]

fig = px.bar(
    outlier_pct,
    x='Feature_kurz',
    y='Ausreisser (%)',
    color='Ausreisser (%)',
    color_continuous_scale='Oranges',
    text='Ausreisser (%)',
    title='Abbildung 6: Top-20-Merkmale mit extremen Ausreissern (mehr als 3-facher IQR-Abstand)'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=450, xaxis_title='Merkmal', coloraxis_showscale=False)
fig.show()

## 4. Data Engineering

Alle Vorverarbeitungsschritte werden konsequent nur auf dem jeweiligen Trainingsset berechnet (fit) und anschliessend auf Validierungs- und Testset uebertragen (transform). Dieses Vorgehen verhindert Data Leakage, also das unbeabsichtigte Einfliesssen von Testinformation in den Trainingsprozess.

### 4.1 Fehlende Werte

Die EDA hat gezeigt, dass Merkmal X37 zu 43,7 % fehlt. Aufgrund dieses hohen Anteils wird X37 vollstaendig aus dem Datensatz entfernt, da jede Imputationsstrategie bei einem derart hohen Fehlanteil zu stark verzerrten Werten fuehren wuerde.

Merkmal X21 fehlt zu 13,5 %. Um den Informationsgehalt des Fehlens selbst zu erhalten, wird ein binaerer Indikator (`X21_missing`) ergaenzt, der anzeigt, ob der urspruengliche Wert fehlte. Anschliessend wird X21 per Median-Imputation aufgefuellt.

Alle uebrigen Merkmale mit weniger als 7 % fehlenden Werten werden ebenfalls per Median-Imputation behandelt. Die Median-Imputation ist robuster gegenueber Ausreissern als die Mittelwert-Imputation und eignet sich daher besonders fuer schiefverteilte Finanzkennzahlen.

### 4.2 Ausreisser

Wie die EDA gezeigt hat, weisen zahlreiche Merkmale extreme Ausreisser auf. Diese sind bei Finanzdaten haeufig keine Messfehler, sondern spiegeln reale wirtschaftliche Extremzustaende wider, die gerade bei insolvenzgefaehrdeten Unternehmen auftreten. Eine Entfernung dieser Werte wuerde daher relevante Information vernichten.

Stattdessen wird ein `RobustScaler` eingesetzt, der auf Median und Interquartilsabstand (IQR) basiert und damit unempfindlich gegenueber extremen Werten ist.

### 4.3 Klassenimbalance

Der Datensatz ist stark unbalanciert: Nur etwa 5 % der Beobachtungen enthoeren der Klasse Bankrott. Ein Klassifikator der naiv immer die Mehrheitsklasse vorhersagt, erraeicht eine Genauigkeit von 95 %, ist aber wertlos.

Um die Minderheitsklasse angemessen zu gewichten, werden klassenbasierte Gewichte eingesetzt (`class_weight='balanced'` fuer Logistische Regression und Random Forest, `scale_pos_weight` fuer XGBoost). Diese Gewichte sind invers proportional zur Klassenhaeufigkeit und erhoehen den Einfluss von Bankrott-Faellen auf den Lernprozess, ohne die reale Verteilung durch synthetische Datenpunkte zu veraendern.

### 4.4 Feature Selection

Aufgrund der hohen Korrelation zwischen vielen Finanzkennzahlen wurde ein korrelationsbasierter Filteransatz (Schwellwert 0,9) evaluiert. Fuer baumbasierte Modelle (Random Forest, XGBoost) ergaben Experimente, dass die Vorhersagequalitaet ohne Feature Selection hoeher ist. Dies ist methodisch erklaerbar: Entscheidungsbaumverfahren waehlen bei jedem Knoten intern das informativste Merkmal und ignorieren redundante Merkmale dadurch automatisch. Eine vorgelagerte Selektion entfernt unter Umstaenden Merkmale, die in Kombination mit anderen Merkmalen nuetzlich waeren. Daher wird im vorliegenden Projekt keine vorgelagerte Feature Selection angewendet.

## 5. Modellierung

### 5.1 Begruendung der Methodenwahl

Drei Klassifikationsverfahren werden eingesetzt, die unterschiedliche Komplexitaetsstufen repraesentieren und damit eine fundierte Vergleichsbasis bilden:

**Logistische Regression** dient als Baseline-Modell. Sie ist linear, interpretierbar und in der Insolvenzforschung seit Ohlson (1980) etabliert.

**Random Forest** ist ein Ensemble-Verfahren das viele unabhaengige Entscheidungsbaeume kombiniert. Es ist robust gegenueber Ausreissern und Multikollinearitaet und kann nichtlineare Zusammenhaenge erfassen.

**XGBoost** (Extreme Gradient Boosting) ist ein sequenzielles Boosting-Verfahren und gilt bei tabellarischen Daten als State-of-the-Art (Barboza et al., 2017).

### 5.2 Evaluationsmetriken

Bei stark unbalancierten Datensaetzen ist Accuracy als Metrik ungeeignet. Verwendet werden: ROC-AUC (schwellenwertunabhaengiger Modellvergleich), F1-Score, Precision und Recall jeweils fuer die Bankrott-Klasse.

### 5.3 Datenaufteilung und Imbalance-Behandlung

Da jeder der fuenf Datensaetze einen eigenstaendigen Prognosehorizont repraesentiert, wird fuer jeden Datensatz ein separates Modell trainiert. Aufgrund des begrenzten Anteils an Bankrott-Faellen (rund 5 Prozent) wird auf ein separates Validierungsset verzichtet. Stattdessen wird ein stratifizierter 85/15-Split (Train/Test) eingesetzt, um dem Modell moeglichst viele Trainingsdaten zur Verfuegung zu stellen.

Die Klassenimbalance wird fuer alle Modelle durch klassenbasierte Gewichte behandelt. Fuer XGBoost wird zusaetzlich `sample_weight` im `fit()`-Aufruf uebergeben, was einer doppelten Gewichtung der Minderheitsklasse entspricht und den Recall erhoehen soll.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.utils import compute_sample_weight
from sklearn.metrics import (roc_auc_score, f1_score,
                             precision_score, recall_score,
                             classification_report, roc_curve,
                             confusion_matrix)
from xgboost import XGBClassifier

In [ ]:
DROP_FEATURE = 'X37_current_assets_inv_long_term_liabilities'
MISSING_IND  = 'X21_sales_growth'
RANDOM_STATE = 42

results        = []
test_sets      = {}
fitted_models  = {}
test_probas    = {}
final_cols_map = {}

for year in range(1, 6):
    df_year = df_all[df_all['year'] == year].copy()
    raw_feat_cols = [c for c in df_year.columns if c not in ['class', 'year']]
    X = df_year[raw_feat_cols]
    y = df_year['class']

    # Stratifizierter 85/15-Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y)

    # X37 entfernen (44% fehlend)
    for Xs in [X_train, X_test]:
        Xs.drop(columns=[DROP_FEATURE], inplace=True, errors='ignore')

    # Binaerer Indikator fuer fehlende X21-Werte
    for Xs in [X_train, X_test]:
        Xs[f'{MISSING_IND}_missing'] = Xs[MISSING_IND].isnull().astype(int)

    num_cols = [c for c in X_train.columns if c != f'{MISSING_IND}_missing']

    # Median-Imputation: fit auf Train, transform auf beide Sets
    imputer = SimpleImputer(strategy='median')
    X_train[num_cols] = imputer.fit_transform(X_train[num_cols])
    X_test[num_cols]  = imputer.transform(X_test[num_cols])

    final_cols = list(X_train.columns)
    final_cols_map[year] = final_cols

    # RobustScaler: fit auf Train, transform auf beide Sets
    scaler    = RobustScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    test_sets[year]     = (X_test_s, y_test)
    fitted_models[year] = {}
    test_probas[year]   = {}

    # Klassengewichte fuer doppelte Imbalance-Behandlung bei XGBoost
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
    pos_weight     = (y_train == 0).sum() / (y_train == 1).sum()

    models = {
        'Logistic Regression': LogisticRegression(
            class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
        'Random Forest': RandomForestClassifier(
            class_weight='balanced', n_estimators=200,
            random_state=RANDOM_STATE, n_jobs=-1),
        'XGBoost': XGBClassifier(
            scale_pos_weight=pos_weight, n_estimators=200,
            random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0),
    }

    # XGBoost erhaelt zusaetzlich sample_weight (doppelte Gewichtung)
    fit_kwargs = {
        'Logistic Regression': {},
        'Random Forest':       {},
        'XGBoost':             {'sample_weight': sample_weights},
    }

    for name, model in models.items():
        model.fit(X_train_s, y_train, **fit_kwargs[name])
        fitted_models[year][name] = model

        y_pred  = model.predict(X_test_s)
        y_proba = model.predict_proba(X_test_s)[:, 1]
        test_probas[year][name] = y_proba

        results.append({
            'Jahr':                 year,
            'Horizont':             f'{6 - year} Jahr(e)',
            'Modell':               name,
            'ROC-AUC':              round(roc_auc_score(y_test, y_proba), 4),
            'F1 (Bankrott)':        round(f1_score(y_test, y_pred, pos_label=1, zero_division=0), 4),
            'Precision (Bankrott)': round(precision_score(y_test, y_pred, pos_label=1, zero_division=0), 4),
            'Recall (Bankrott)':    round(recall_score(y_test, y_pred, pos_label=1, zero_division=0), 4),
        })

    print(f'Jahr {year}: Train={len(y_train)}, Test={len(y_test)}, Merkmale={len(final_cols)}')

results_df = pd.DataFrame(results)
print('\nTraining abgeschlossen.')
results_df

## 6. Evaluation (Testset)

Die Modelle wurden auf 85 Prozent der Daten trainiert. Die folgenden Ergebnisse beziehen sich auf das zurueckgehaltene Testset (15 Prozent), das zu keinem Zeitpunkt in den Trainingsprozess eingeflossen ist.

In [ ]:
horizon_order = ['5 Jahr(e)', '4 Jahr(e)', '3 Jahr(e)', '2 Jahr(e)', '1 Jahr(e)']

for metric in ['ROC-AUC', 'F1 (Bankrott)', 'Precision (Bankrott)', 'Recall (Bankrott)']:
    fig = px.bar(
        results_df,
        x='Horizont', y=metric, color='Modell',
        barmode='group', text=metric,
        title=f'Abbildung 7: {metric} je Modell und Prognosehorizont (Testset)',
        category_orders={'Horizont': horizon_order}
    )
    fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
    fig.update_layout(height=430)
    fig.show()

In [ ]:
for year in range(1, 6):
    X_test_s, y_test = test_sets[year]
    roc_rows = []
    for name in ['Logistic Regression', 'Random Forest', 'XGBoost']:
        fpr, tpr, _ = roc_curve(y_test, test_probas[year][name])
        auc = roc_auc_score(y_test, test_probas[year][name])
        for fp, tp in zip(fpr, tpr):
            roc_rows.append({'FPR': fp, 'TPR': tp, 'Modell': f'{name} (AUC={auc:.3f})'})
    roc_df = pd.DataFrame(roc_rows)
    fig = px.line(
        roc_df, x='FPR', y='TPR', color='Modell',
        title=f'Abbildung 9: ROC-Kurve, Jahr {year} ({6-year} Jahr(e) Horizont, Testset)',
        labels={'FPR': 'False Positive Rate', 'TPR': 'True Positive Rate'}
    )
    fig.add_shape(type='line', x0=0, y0=0, x1=1, y1=1, line=dict(dash='dash', color='grey'))
    fig.update_layout(height=430)
    fig.show()

In [ ]:
for year in range(1, 6):
    X_test_s, y_test = test_sets[year]
    model  = fitted_models[year]['XGBoost']
    y_pred = model.predict(X_test_s)
    cm     = confusion_matrix(y_test, y_pred)
    labels = ['Nicht bankrott (0)', 'Bankrott (1)']
    cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
    text   = [[f'{cm[i][j]}<br>({cm_pct[i][j]:.1f}%)' for j in range(2)] for i in range(2)]
    fig = px.imshow(
        cm_pct, x=labels, y=labels,
        color_continuous_scale='Blues', zmin=0, zmax=100,
        title=f'Abbildung 10: Konfusionsmatrix XGBoost, Jahr {year} ({6-year} Jahr(e) Horizont)',
        labels={'x': 'Vorhergesagt', 'y': 'Tatsaechlich', 'color': '%'}
    )
    for i in range(2):
        for j in range(2):
            fig.add_annotation(x=j, y=i, text=text[i][j], showarrow=False,
                               font=dict(size=14, color='black'))
    fig.update_layout(height=380, coloraxis_showscale=False)
    fig.show()

In [ ]:
model_xgb = fitted_models[5]['XGBoost']
cols      = final_cols_map[5]

imp_df = pd.DataFrame({
    'Feature':      [c.split('_')[0] for c in cols],
    'Importance':   model_xgb.feature_importances_,
    'Feature_voll': cols
}).sort_values('Importance', ascending=True).tail(20)

fig = px.bar(
    imp_df,
    x='Importance', y='Feature',
    orientation='h',
    color='Importance',
    color_continuous_scale='Blues',
    hover_data={'Feature_voll': True, 'Feature': False},
    title='Abbildung 11: Feature Importance XGBoost, Jahr 5 (Top 20, 1 Jahr Horizont)'
)
fig.update_layout(height=550, coloraxis_showscale=False,
                  yaxis_title='Merkmal', xaxis_title='Importance Score')
fig.show()

In [ ]:
X_test_s, y_test = test_sets[5]
print('Classification Report: XGBoost, Jahr 5 (1 Jahr Horizont, Testset)')
print(classification_report(y_test, fitted_models[5]['XGBoost'].predict(X_test_s),
                            target_names=['Nicht bankrott', 'Bankrott']))

## 8. Diskussion und Fazit

### 8.1 Zentrale Befunde

Die Ergebnisse zeigen, dass XGBoost den beiden anderen Verfahren konsistent ueberlegen ist. Dies deckt sich mit den Befunden von Barboza et al. (2017), die baumbasierte Boosting-Verfahren bei tabellarischen Finanzdaten als ueberlegen identifizierten. Die Logistische Regression erzielt hohe Recall-Werte, produziert jedoch sehr viele Fehlalarme (niedrige Precision). Random Forest ist hochpraezise, erkennt aber nur einen kleinen Teil der tatsaechlichen Insolvenzen.

Der Prognosehorizont hat erwartungsgemaess Einfluss auf die Modellguete: XGBoost erzielt beim 1-Jahres-Horizont (Jahr 5) mit einem ROC-AUC von circa 0,94 und einem F1-Score von circa 0,64 die besten Ergebnisse. Bei laengeren Horizonten sinkt die Vorhersagequalitaet, was inhaltlich plausibel ist: Finanzkennzahlen 5 Jahre vor einer Insolvenz enthalten weniger Vorabinformation ueber ein bevorstehendes Scheitern.

Die Threshold-Optimierung in Kombination mit Hyperparameter-Tuning verbessert den F1-Score gegenueber dem Standardmodell, insbesondere bei mittleren und laengeren Horizonten.

Die Feature-Importance-Analyse zeigt, dass Profitabilitaets- und Verschuldungskennzahlen (u.a. X1, X2, X7, X10) die hoeChste Bedeutsamkeit aufweisen. Dies entspricht dem theoretischen Hintergrund: Ein Unternehmen auf dem Weg in die Insolvenz ist typischerweise nicht mehr profitabel und weist eine erhoehte Verschuldung auf.

### 8.2 Limitationen

Das vorliegende Modell weist mehrere Limitationen auf, die bei der Interpretation der Ergebnisse zu beruecksichtigen sind:

1. **Zeitliche Struktur:** Die Aufteilung in Train, Validation und Test erfolgte zufaellig. Eine zeitreihenbasierte Aufteilung (z.B. Training auf aelteren, Test auf neueren Daten) wuerde die reale Prognosesituation besser abbilden.

2. **Mehrfacherfassung:** Einzelne Unternehmen koennen in mehreren Jahresdatensaetzen auftreten. Da die Modelle je Jahr separat trainiert werden, ist dies bei der jahruebergreifenden Interpretation zu beachten.

3. **Geografische Spezifitaet:** Die Daten stammen ausschliesslich aus dem polnischen Markt. Die Uebertragbarkeit auf andere Laender oder Wirtschaftssysteme ist nicht gesichert.

4. **Fehlende makrooekonomische Faktoren:** Das Modell beruecksichtigt ausschliesslich unternehmensinterne Kennzahlen. Makrooekonomische Kontextfaktoren wie Konjunkturphasen oder Zinsniveaus koennen die Insolvenzwahrscheinlichkeit beeinflussen, sind aber im Datensatz nicht enthalten.

### 8.3 Fazit

Die wissenschaftliche Fragestellung kann positiv beantwortet werden: Machine-Learning-Modelle, insbesondere XGBoost, sind in der Lage, Insolvenzrisiken polnischer Unternehmen auf Basis von Jahresabschlussdaten mit hoher Guete vorherzusagen. Der ROC-AUC-Wert von circa 0,94 beim kurzfristigen Horizont weist auf eine sehr gute Trennfaehigkeit hin. Der F1-Score von circa 0,64 zeigt jedoch auch, dass ein substanzieller Anteil der Insolvenzen noch nicht korrekt erkannt wird, was auf die inhaerent schwierige Vorhersage seltener Ereignisse bei unbalancierten Datensaetzen zurueckzufuehren ist.

## Literaturverzeichnis

Altman, E.I. (1968). Financial ratios, discriminant analysis and the prediction of corporate bankruptcy. *Journal of Finance*, 23(4), 589-609.

Barboza, F., Kimura, H., & Altman, E. (2017). Machine learning models and bankruptcy prediction. *Expert Systems with Applications*, 83, 405-417.

Beaver, W.H. (1966). Financial ratios as predictors of failure. *Journal of Accounting Research*, 4, 71-111.

Liang, D., Lu, C.C., Tsai, C.F., & Shih, G.A. (2016). Financial ratios and corporate governance indicators in bankruptcy prediction. *Expert Systems with Applications*, 52, 227-240.

Ohlson, J.A. (1980). Financial ratios and the probabilistic prediction of bankruptcy. *Journal of Accounting Research*, 18(1), 109-131.

Zieba, M., Tomczak, S.K., & Tomczak, J.M. (2016). Ensemble boosted trees with synthetic features generation in application to bankruptcy prediction. *Expert Systems with Applications*, 58, 93-101.